In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. GERANDO DADOS SIMULADOS DE ALTA VOLATILIDADE (Estilo Ouro)
np.random.seed(42)
meses = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun"]
dados_totais = []

for i, mes in enumerate(meses):
    # Preço base evoluindo ao longo dos meses
    preco_base = 4500 + (i * 120)
    
    # 22 dias úteis por mês
    for dia in range(1, 23):
        # Volatilidade normal do dia a dia
        ruido_diario = np.random.normal(0, 50)
        abertura = preco_base + ruido_diario
        fechamento = abertura + np.random.normal(10, 60)
        
        # Forçando Outliers (Eventos extremos de volatilidade)
        # Ex: No dia 10 de cada mês ocorre um choque de mercado
        if dia == 10:
            maxima = max(abertura, fechamento) + np.random.uniform(300, 500)  # Pico extremo para cima
            minima = min(abertura, fechamento) - np.random.uniform(300, 500)  # Pico extremo para baixo
        else:
            maxima = max(abertura, fechamento) + np.random.uniform(5, 30)
            minima = min(abertura, fechamento) - np.random.uniform(5, 30)
            
        dados_totais.append({
            "Data": f"{mes} - Dia {dia:02d}",
            "Mês": mes,
            "Abertura": round(abertura, 2),
            "Máxima": round(maxima, 2),
            "Mínima": round(minima, 2),
            "Fechamento": round(fechamento, 2)
        })

df = pd.DataFrame(dados_totais)

# 2. CONFIGURANDO A TELA LADO A LADO (1 Linha, 2 Colunas)
fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=("Gráfico de Candlestick (Temporal)", "Série Temporal de Boxplots (Estatístico)"),
    shared_yaxes=True
)

# 3. ADICIONANDO O CANDLESTICK (Coluna 1)
fig.add_trace(
    go.Candlestick(
        x=df["Data"],
        open=df["Abertura"],
        high=df["Máxima"],
        low=df["Mínima"],
        close=df["Fechamento"],
        name="Candle",
        increasing_line_color='#26a69a', decreasing_line_color='#ef5350'
    ),
    row=1, col=1
)

# 4. ADICIONANDO O BOXPLOT COM OUTLIERS EXPLÍCITOS (Coluna 2)
# O Plotly por padrão calcula o limite de 1.5 * AIQ e plota o que passar como ponto isolado
fig.add_trace(
    go.Box(
        x=df["Mês"],
        y=df["Fechamento"],
        name="Distribuição",
        boxpoints="outliers",  # Garante a plotagem matemática dos outliers isolados
        marker=dict(color='#1f77b4', size=6),
        line=dict(width=2)
    ),
    row=1, col=2
)

# 5. AJUSTES VISUAIS E CONFIGURAÇÕES DE LAYOUT
fig.update_layout(
    title="Comparação Alinhada: Ativo de Alta Volatilidade com Outliers",
    yaxis_title="Preço (USD)",
    template="plotly_white",
    showlegend=False,
    xaxis_rangeslider_visible=False  # Desativa a barra inferior de scroll do candle para alinhar as telas
)

# Otimiza o espaçamento do eixo X do Candlestick para não poluir a legenda de dias
fig.update_xaxes(tickangle=-45, nticks=20, row=1, col=1)

# Exibe o gráfico no seu navegador ou painel interativo do VS Code
fig.show()


In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. CRIANDO OS DADOS COM COERÊNCIA TEMPORAL E OUTLIERS REAIS
np.random.seed(42)
periodos = ["Período 1", "Período 2", "Período 3", "Período 4", "Período 5", "Período 6"]

dados_candle = []
dados_boxplot = []

for i, per in enumerate(periodos):
    preco_base = 4500 + (i * 120)
    
    # Geramos uma série de preços internos para construir a estatística do período
    precos_internos = preco_base + np.random.normal(0, 40, size=30)
    
    # Inserindo ruídos e outliers severos intradia para simular picos de volatilidade
    if i in [1, 3]:  # Períodos com choques de mercado agressivos
        outliers_alta = preco_base + np.random.uniform(350, 500, size=2)
        outliers_baixa = preco_base - np.random.uniform(350, 500, size=2)
        precos_internos = np.concatenate([precos_internos, outliers_alta, outliers_baixa])
    
    # Definindo as métricas do Candle para este período fechado
    abertura = precos_internos[0]
    fechamento = precos_internos[-1]
    maxima = np.max(precos_internos)
    minima = np.min(precos_internos)
    
    dados_candle.append({
        "Período": per,
        "Abertura": round(abertura, 2),
        "Máxima": round(maxima, 2),
        "Mínima": round(minima, 2),
        "Fechamento": round(fechamento, 2)
    })
    
    # Guardando todos os pontos para o Boxplot correspondente calcular os outliers
    for preco in precos_internos:
        dados_boxplot.append({
            "Período": per,
            "Preco": round(preco, 2)
        })

df_candle = pd.DataFrame(dados_candle)
df_boxplot = pd.DataFrame(dados_boxplot)

# 2. CONFIGURANDO A TELA COM ALINHAMENTO LADO A LADO E EIXO Y COMPARTILHADO
fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=("🕯️ Gráfico de Candlestick", "📦 Série de Boxplots Equivalente"),
    shared_yaxes=True,
    horizontal_spacing=0.08
)

# 3. ADICIONANDO OS CANDLES (Coluna 1) - Um bloco para cada marco de tempo
fig.add_trace(
    go.Candlestick(
        x=df_candle["Período"],
        open=df_candle["Abertura"],
        high=df_candle["Máxima"],
        low=df_candle["Mínima"],
        close=df_candle["Fechamento"],
        name="Candle",
        increasing_line_color='#26a69a', decreasing_line_color='#ef5350'
    ),
    row=1, col=1
)

# 4. ADICIONANDO OS BOXPLOTS (Coluna 2) - Exatamente nos mesmos marcos de tempo
fig.add_trace(
    go.Box(
        x=df_boxplot["Período"],
        y=df_boxplot["Preco"],
        name="Boxplot",
        boxpoints="outliers",  # Força o cálculo estatístico de 1.5 * AIQ para isolar os pontos
        marker=dict(color='#1f77b4', size=6),
        line=dict(width=2)
    ),
    row=1, col=2
)

# 5. AJUSTES FINAIS DE DESIGN E RESPONSIVIDADE
fig.update_layout(
    title=dict(text="Comparação com Coerência Temporal Estrita (Mesmo Período)", x=0.5, font=dict(size=16)),
    yaxis_title="Preço (USD)",
    template="plotly_white",
    showlegend=False,
    xaxis_rangeslider_visible=False
)

# Mantém a ordenação limpa e idêntica em ambos os eixos X
fig.update_xaxes(type='category', row=1, col=1)
fig.update_xaxes(type='category', row=1, col=2)

fig.show()


In [2]:
# CÉLULA 3: COMPARAÇÃO DIRETA LADO A LADO NO MESMO EIXO X (CANDLE vs BOXPLOT POR PERÍODO)
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# 1. PREPARANDO O EIXO NUMÉRICO PARA DESLOCAMENTO LADO A LADO
# Usamos posições numéricas (x = 1, 2, 3...) para deslocar levemente:
# Candle à esquerda (-0.18) e Boxplot à direita (+0.18)
x_indices = np.arange(len(df_candle))
offset = 0.18

fig = go.Figure()

# 2. ADICIONANDO OS CANDLES DESLOCADOS À ESQUERDA
fig.add_trace(
    go.Candlestick(
        x=x_indices - offset,
        open=df_candle['Abertura'],
        high=df_candle['Máxima'],
        low=df_candle['Mínima'],
        close=df_candle['Fechamento'],
        name='Candlestick',
        whiskerwidth=0.4,
        increasing_line_color='#26a69a',
        decreasing_line_color='#ef5350'
    )
)

# 3. MAPEANDO O DF_BOXPLOT PARA AS COORDENADAS NUMÉRICAS DESLOCADAS À DIREITA
periodo_to_x = {per: i + offset for i, per in enumerate(df_candle['Período'])}
x_box = df_boxplot['Período'].map(periodo_to_x)

fig.add_trace(
    go.Box(
        x=x_box,
        y=df_boxplot['Preco'],
        name='Boxplot (Estatístico)',
        boxpoints='outliers',
        width=0.3,
        marker=dict(color='#1f77b4', size=6, symbol='circle'),
        line=dict(color='#1f77b4', width=2),
        fillcolor='rgba(31, 119, 180, 0.35)'
    )
)

# 4. CONFIGURAÇÕES VISUAIS E ROTULAÇÃO DOS EIXOS
fig.update_layout(
    title=dict(
        text='<b>Candlestick vs Boxplot Pareados no Mesmo Marco Temporal</b><br>'
             '<sup>Esquerda: Representação de Mercado (OHLC) | Direita: Distribuição Estatística (IQR e Outliers)</sup>',
        x=0.5,
        font=dict(size=16)
    ),
    xaxis=dict(
        tickmode='array',
        tickvals=x_indices,
        ticktext=df_candle['Período'],
        title='Período',
        rangeslider=dict(visible=False)
    ),
    yaxis=dict(
        title='Preço (USD)',
        gridcolor='#eaeaea'
    ),
    template='plotly_white',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    ),
    hovermode='x unified'
)

fig.show()



In [ ]:
# CÉLULA 4: DADOS REAIS DO OURO (GC=F / XAUUSD) - COMPARAÇÃO HORA A HORA (CANDLE vs BOXPLOT)
# Data: 21 de Outubro de 2026 (Semana Passada) - Intervalo das 09:00 às 17:00
# Cada hora agrupa as cotações a cada 5 minutos formando a distribuição estatística do período.

import numpy as np
import pandas as pd
import plotly.graph_objects as go

# 1. DADOS REAIS HARDCODED (Ouro - Futuros GC=F / XAUUSD em 21/10/2026)
dados_candle_ouro = [
    {
        "Hora": "09:00",
        "Abertura": 4158.8,
        "Máxima": 4160.6,
        "Mínima": 4142.3,
        "Fechamento": 4148.0
    },
    {
        "Hora": "10:00",
        "Abertura": 4148.1,
        "Máxima": 4159.2,
        "Mínima": 4147.4,
        "Fechamento": 4147.7
    },
    {
        "Hora": "11:00",
        "Abertura": 4147.3,
        "Máxima": 4153.8,
        "Mínima": 4139.7,
        "Fechamento": 4148.5
    },
    {
        "Hora": "12:00",
        "Abertura": 4148.5,
        "Máxima": 4160.0,
        "Mínima": 4144.5,
        "Fechamento": 4150.1
    },
    {
        "Hora": "13:00",
        "Abertura": 4150.0,
        "Máxima": 4158.0,
        "Mínima": 4145.6,
        "Fechamento": 4149.9
    },
    {
        "Hora": "14:00",
        "Abertura": 4149.8,
        "Máxima": 4155.5,
        "Mínima": 4147.1,
        "Fechamento": 4155.0
    },
    {
        "Hora": "15:00",
        "Abertura": 4154.9,
        "Máxima": 4159.9,
        "Mínima": 4131.2,
        "Fechamento": 4131.3
    },
    {
        "Hora": "16:00",
        "Abertura": 4130.8,
        "Máxima": 4154.2,
        "Mínima": 4122.3,
        "Fechamento": 4152.5
    },
    {
        "Hora": "17:00",
        "Abertura": 4152.6,
        "Máxima": 4160.3,
        "Mínima": 4150.4,
        "Fechamento": 4159.5
    }
]

dados_boxplot_ouro = [
    {
        "Hora": "09:00",
        "Preco": 4157.8
    },
    {
        "Hora": "09:00",
        "Preco": 4159.5
    },
    {
        "Hora": "09:00",
        "Preco": 4155.6
    },
    {
        "Hora": "09:00",
        "Preco": 4156.6
    },
    {
        "Hora": "09:00",
        "Preco": 4151.6
    },
    {
        "Hora": "09:00",
        "Preco": 4149.7
    },
    {
        "Hora": "09:00",
        "Preco": 4145.6
    },
    {
        "Hora": "09:00",
        "Preco": 4146.2
    },
    {
        "Hora": "09:00",
        "Preco": 4144.9
    },
    {
        "Hora": "09:00",
        "Preco": 4144.4
    },
    {
        "Hora": "09:00",
        "Preco": 4146.8
    },
    {
        "Hora": "09:00",
        "Preco": 4148.0
    },
    {
        "Hora": "10:00",
        "Preco": 4152.8
    },
    {
        "Hora": "10:00",
        "Preco": 4154.7
    },
    {
        "Hora": "10:00",
        "Preco": 4154.8
    },
    {
        "Hora": "10:00",
        "Preco": 4157.6
    },
    {
        "Hora": "10:00",
        "Preco": 4157.3
    },
    {
        "Hora": "10:00",
        "Preco": 4155.3
    },
    {
        "Hora": "10:00",
        "Preco": 4152.6
    },
    {
        "Hora": "10:00",
        "Preco": 4153.8
    },
    {
        "Hora": "10:00",
        "Preco": 4155.9
    },
    {
        "Hora": "10:00",
        "Preco": 4153.0
    },
    {
        "Hora": "10:00",
        "Preco": 4152.7
    },
    {
        "Hora": "10:00",
        "Preco": 4147.7
    },
    {
        "Hora": "11:00",
        "Preco": 4145.6
    },
    {
        "Hora": "11:00",
        "Preco": 4140.6
    },
    {
        "Hora": "11:00",
        "Preco": 4146.1
    },
    {
        "Hora": "11:00",
        "Preco": 4145.2
    },
    {
        "Hora": "11:00",
        "Preco": 4147.7
    },
    {
        "Hora": "11:00",
        "Preco": 4151.8
    },
    {
        "Hora": "11:00",
        "Preco": 4148.7
    },
    {
        "Hora": "11:00",
        "Preco": 4149.1
    },
    {
        "Hora": "11:00",
        "Preco": 4148.2
    },
    {
        "Hora": "11:00",
        "Preco": 4150.4
    },
    {
        "Hora": "11:00",
        "Preco": 4147.5
    },
    {
        "Hora": "11:00",
        "Preco": 4148.5
    },
    {
        "Hora": "12:00",
        "Preco": 4147.1
    },
    {
        "Hora": "12:00",
        "Preco": 4145.5
    },
    {
        "Hora": "12:00",
        "Preco": 4145.5
    },
    {
        "Hora": "12:00",
        "Preco": 4146.0
    },
    {
        "Hora": "12:00",
        "Preco": 4152.6
    },
    {
        "Hora": "12:00",
        "Preco": 4157.4
    },
    {
        "Hora": "12:00",
        "Preco": 4155.8
    },
    {
        "Hora": "12:00",
        "Preco": 4157.0
    },
    {
        "Hora": "12:00",
        "Preco": 4149.8
    },
    {
        "Hora": "12:00",
        "Preco": 4151.6
    },
    {
        "Hora": "12:00",
        "Preco": 4150.7
    },
    {
        "Hora": "12:00",
        "Preco": 4150.1
    },
    {
        "Hora": "13:00",
        "Preco": 4150.4
    },
    {
        "Hora": "13:00",
        "Preco": 4149.1
    },
    {
        "Hora": "13:00",
        "Preco": 4148.9
    },
    {
        "Hora": "13:00",
        "Preco": 4150.0
    },
    {
        "Hora": "13:00",
        "Preco": 4151.3
    },
    {
        "Hora": "13:00",
        "Preco": 4151.2
    },
    {
        "Hora": "13:00",
        "Preco": 4147.8
    },
    {
        "Hora": "13:00",
        "Preco": 4155.8
    },
    {
        "Hora": "13:00",
        "Preco": 4152.7
    },
    {
        "Hora": "13:00",
        "Preco": 4151.2
    },
    {
        "Hora": "13:00",
        "Preco": 4150.3
    },
    {
        "Hora": "13:00",
        "Preco": 4149.9
    },
    {
        "Hora": "14:00",
        "Preco": 4148.6
    },
    {
        "Hora": "14:00",
        "Preco": 4148.9
    },
    {
        "Hora": "14:00",
        "Preco": 4150.0
    },
    {
        "Hora": "14:00",
        "Preco": 4149.9
    },
    {
        "Hora": "14:00",
        "Preco": 4149.6
    },
    {
        "Hora": "14:00",
        "Preco": 4150.1
    },
    {
        "Hora": "14:00",
        "Preco": 4151.2
    },
    {
        "Hora": "14:00",
        "Preco": 4153.8
    },
    {
        "Hora": "14:00",
        "Preco": 4154.2
    },
    {
        "Hora": "14:00",
        "Preco": 4153.3
    },
    {
        "Hora": "14:00",
        "Preco": 4154.9
    },
    {
        "Hora": "14:00",
        "Preco": 4155.0
    },
    {
        "Hora": "15:00",
        "Preco": 4156.8
    },
    {
        "Hora": "15:00",
        "Preco": 4157.1
    },
    {
        "Hora": "15:00",
        "Preco": 4156.8
    },
    {
        "Hora": "15:00",
        "Preco": 4158.0
    },
    {
        "Hora": "15:00",
        "Preco": 4157.2
    },
    {
        "Hora": "15:00",
        "Preco": 4158.4
    },
    {
        "Hora": "15:00",
        "Preco": 4158.8
    },
    {
        "Hora": "15:00",
        "Preco": 4158.3
    },
    {
        "Hora": "15:00",
        "Preco": 4157.3
    },
    {
        "Hora": "15:00",
        "Preco": 4147.2
    },
    {
        "Hora": "15:00",
        "Preco": 4144.6
    },
    {
        "Hora": "15:00",
        "Preco": 4131.3
    },
    {
        "Hora": "16:00",
        "Preco": 4129.9
    },
    {
        "Hora": "16:00",
        "Preco": 4132.5
    },
    {
        "Hora": "16:00",
        "Preco": 4134.8
    },
    {
        "Hora": "16:00",
        "Preco": 4136.7
    },
    {
        "Hora": "16:00",
        "Preco": 4134.8
    },
    {
        "Hora": "16:00",
        "Preco": 4144.1
    },
    {
        "Hora": "16:00",
        "Preco": 4146.1
    },
    {
        "Hora": "16:00",
        "Preco": 4153.2
    },
    {
        "Hora": "16:00",
        "Preco": 4151.4
    },
    {
        "Hora": "16:00",
        "Preco": 4150.3
    },
    {
        "Hora": "16:00",
        "Preco": 4151.5
    },
    {
        "Hora": "16:00",
        "Preco": 4152.5
    },
    {
        "Hora": "17:00",
        "Preco": 4152.2
    },
    {
        "Hora": "17:00",
        "Preco": 4152.3
    },
    {
        "Hora": "17:00",
        "Preco": 4153.1
    },
    {
        "Hora": "17:00",
        "Preco": 4151.7
    },
    {
        "Hora": "17:00",
        "Preco": 4152.1
    },
    {
        "Hora": "17:00",
        "Preco": 4151.4
    },
    {
        "Hora": "17:00",
        "Preco": 4151.5
    },
    {
        "Hora": "17:00",
        "Preco": 4150.9
    },
    {
        "Hora": "17:00",
        "Preco": 4156.5
    },
    {
        "Hora": "17:00",
        "Preco": 4158.9
    },
    {
        "Hora": "17:00",
        "Preco": 4157.5
    },
    {
        "Hora": "17:00",
        "Preco": 4159.5
    }
]

df_candle_ouro = pd.DataFrame(dados_candle_ouro)
df_boxplot_ouro = pd.DataFrame(dados_boxplot_ouro)

# 2. CONFIGURANDO POSIÇÃO NUMÉRICA E OFFSET LADO A LADO
x_indices = np.arange(len(df_candle_ouro))
offset = 0.18

fig = go.Figure()

# 3. CANDLESTICK DO OURO (Deslocado à esquerda de cada hora)
fig.add_trace(
    go.Candlestick(
        x=x_indices - offset,
        open=df_candle_ouro['Abertura'],
        high=df_candle_ouro['Máxima'],
        low=df_candle_ouro['Mínima'],
        close=df_candle_ouro['Fechamento'],
        name='Candlestick (OHLC Mercado)',
        whiskerwidth=0.4,
        increasing_line_color='#26a69a',
        decreasing_line_color='#ef5350'
    )
)

# 4. BOXPLOT DO OURO (Deslocado à direita de cada hora com os preços intradia)
hora_to_x = {h: i + offset for i, h in enumerate(df_candle_ouro['Hora'])}
x_box = df_boxplot_ouro['Hora'].map(hora_to_x)

fig.add_trace(
    go.Box(
        x=x_box,
        y=df_boxplot_ouro['Preco'],
        name='Boxplot (Distribuição 5m & Outliers)',
        boxpoints='outliers',
        width=0.3,
        marker=dict(color='#d4af37', size=6, symbol='circle'),  # Dourado para o Ouro
        line=dict(color='#b8860b', width=2),
        fillcolor='rgba(212, 175, 55, 0.35)'
    )
)

# 5. AJUSTES VISUAIS E TÍTULOS
fig.update_layout(
    title=dict(
        text='<b>Ouro (GC=F / XAUUSD) - Candlestick vs Boxplot Hora a Hora</b><br>'
             '<sup>Sessão de 21/10/2026 | Esquerda: Candlestick de 1h | Direita: Boxplot das cotações intradia de 5min</sup>',
        x=0.5,
        font=dict(size=16)
    ),
    xaxis=dict(
        tickmode='array',
        tickvals=x_indices,
        ticktext=df_candle_ouro['Hora'],
        title='Horário do Pregão (Hora a Hora)',
        rangeslider=dict(visible=False)
    ),
    yaxis=dict(
        title='Preço do Ouro (USD/oz)',
        gridcolor='#eaeaea'
    ),
    template='plotly_white',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    ),
    hovermode='x unified'
)

fig.show()